# TP 1 — Environnement, Git et les limites du traitement local

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

Ce notebook guide les parties **C** (génération des données), **D** (exploration
Pandas) et **E** (montée en charge) du TP 1. Complétez les cellules marquées
`À COMPLÉTER`, exécutez tout de bout en bout, puis poussez le notebook **avec
ses sorties** dans `notebooks/` de votre dépôt Git.

> Réflexe d'ingénieur : on ne dit pas « c'est lent », on dit « 12,4 s pour
> 50 000 lignes ». **Toutes** vos observations doivent être chiffrées.

## 0. Vérification de l'environnement

La cellule suivante doit s'exécuter sans erreur et afficher des versions
cohérentes (Python ≥ 3.9). Sinon, retournez à la partie A du TP
(`python3 check_env.py`).

In [2]:
import sys, platform
import pandas as pd
import numpy as np

print("Python :", sys.version.split()[0], "—", platform.system())
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

Python : 3.13.5 — Windows
pandas : 3.0.4
numpy  : 2.5.0


## 1. Génération du jeu de données fil rouge (échelle 0.1)

La génération est **déterministe** (graine 42) : toute la promotion obtient
exactement les mêmes fichiers. Exécutez la cellule ci-dessous **une seule
fois** (≈ 1 min). Le dossier `data/` est exclu du dépôt Git par le
`.gitignore` fourni — vérifiez-le avec `git status`.

In [3]:
# Depuis la racine du dépôt (adapter le chemin si besoin)
!python ../data/generate_data.py --scale 0.1 --outdir ../data
!dir ../data

Génération (graine=42, échelle=0.1) vers ../data/
  - customers : 5,000 lignes
  - products  : 632 lignes
  - orders    : 50,000 lignes (+ lignes de commandes)
  - payments  : ~44,079 lignes (JSON Lines)
  - events    : ~330,000 lignes (JSON Lines, écrit par lots)
  - pg_init.sql : référentiel propre (customers, products)

Terminé. Fichiers générés :
  anomalies_manifest.json             0.0 Mo
  customers.csv                       0.6 Mo
  events.json                        68.5 Mo
  generate_data.py                    0.0 Mo
  order_items.csv                     3.9 Mo
  orders.csv                          3.3 Mo
  payments.json                       9.0 Mo
  pg_init.sql                         0.6 Mo
  products.csv                        0.0 Mo


Le format du param�tre est incorrect - "data".


### Tableau de relevés

Vous remplirez ce dictionnaire au fil des exercices ; il sert de source unique
pour la partie E et pour votre `DIAGNOSTIC.md`.

In [5]:
DATA_DIR = "../data"

releves = {
    # fichier            : {"lignes": ..., "temps_s": ..., "memoire_Mo": ..., "disque_Mo": ...}
    "customers.csv"      : {},
    "orders.csv"         : {},
    "orders+items (jointure)" : {},
    "events.json"        : {},
}

## Exercice 1 — `customers.csv` : premier contact

Chargez le fichier, puis relevez :
1. ses **dimensions** (lignes × colonnes) ;
2. le **type** de chaque colonne (`dtypes`) ;
3. un **aperçu** (`head`) ;
4. son **empreinte mémoire** en Mo — utilisez
   `df.memory_usage(deep=True).sum()` (pourquoi `deep=True` ? voyez la
   question 1.b).

In [6]:
# === À COMPLÉTER ===
import os, time

t0 = time.perf_counter()
customers = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"))                     # charger DATA_DIR/customers.csv
t1 = time.perf_counter()

print("Dimensions :", customers.shape)
print("Temps de chargement : %.2f s" % (t1 - t0))
print(customers.dtypes)
customers.head()

Dimensions : (5025, 10)
Temps de chargement : 0.09 s
customer_id         str
prenom              str
nom                 str
email               str
telephone           str
adresse             str
ville               str
region              str
date_naissance      str
date_inscription    str
dtype: object


,customer_id,prenom,nom,email,telephone,adresse,ville,region,date_naissance,date_inscription
0,C000878,Yacine,Faye,yacine.faye762@ucad.edu.sn,+221 75 447 17 56,"855, chemin Philippe Grondin",Dakar,Dakar,1963-10-16,2022-10-07
1,C004886,Moussa,Camara,moussa.camara648@orange.sn,+221772902788,"9, chemin de Ribeiro",Louga,Louga,1960-12-03,2025-07-25
2,C002485,Astou,Ndiaye,astou.ndiaye505@orange.sn,+221708464496,"41, chemin Hugues Navarro",Dakar,Dakar,1982-03-01,2022-05-20
3,C002289,Fatoumata,Cissé,fatoumata.cisse59@hotmail.com,77-780-83-93,"98, rue Philippine Hervé",Mbour,Thiès,1993-12-04,2022-02-06
4,C000812,Diarra,Wade,diarra.wade888@yahoo.fr,786934454,"33, chemin de Hoarau",Dakar,Dakar,1966-05-21,2023-04-19


**Question 1.a** — Remplissez la ligne `customers.csv` du tableau de relevés.

In [7]:
# === À COMPLÉTER ===
mem_Mo = customers.memory_usage(deep=True).sum() / 1024**2                       # mémoire du DataFrame en Mo (deep=True)
disque_Mo = os.path.getsize(os.path.join(DATA_DIR, "customers.csv")) / 1024**2    # taille du fichier sur disque en Mo
releves["customers.csv"] = {"lignes": len(customers), "temps_s": round(t1 - t0, 2),
                            "memoire_Mo": round(mem_Mo, 1), "disque_Mo": round(disque_Mo, 1)}
releves["customers.csv"]

{'lignes': 5025,
 'temps_s': 0.09,
 'memoire_Mo': np.float64(2.9),
 'disque_Mo': 0.6}

**Question 1.b** *(markdown — répondez ici)* — Comparez
`memory_usage()` **sans** puis **avec** `deep=True` sur `customers`.
Pourquoi un tel écart ? Quel est le rapport mémoire/disque ? Notez votre
explication : elle resservira mot pour mot dans le diagnostic.

*Votre réponse :* Sans `deep=True`, `memory_usage()` ne comptabilise pas complètement la mémoire occupée par les chaînes de texte et les objets Python. Avec `deep=True`, elle prend en compte la taille réelle de ces valeurs, ce qui explique l’écart important. La mémoire utilisée par le DataFrame est donc bien supérieure à la taille du fichier sur disque, car le CSV est stocké en texte compact alors que Pandas charge les données dans des structures Python plus lourdes. Le rapport mémoire/disque est donc supérieur à 1, environ 5 fois plus de mémoire que le fichier sur disque.

## Exercice 2 — `orders.csv` : statistiques et première surprise

Chargez `orders.csv` en **chronométrant**, puis :
1. nombre de commandes par `statut` ;
2. nombre de commandes par `canal` ;
3. essayez de calculer le **montant total** des commandes
   (`montant_total_fcfa`)… que se passe-t-il ? Regardez le `dtype` de la
   colonne. **Ne corrigez pas** : documentez (section 6).

In [8]:
# === À COMPLÉTER ===
import os, time

t0 = time.perf_counter()
orders = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))  # charger DATA_DIR/orders.csv
t1 = time.perf_counter()
print("Temps : %.2f s — %d lignes" % (t1 - t0, len(orders)))

print("Commandes par statut :")
print(orders["statut"].value_counts())

print("\nCommandes par canal :")
print(orders["canal"].value_counts())

print("\nType de montant_total_fcfa :", orders["montant_total_fcfa"].dtype)
try:
    print("Somme :", orders["montant_total_fcfa"].sum())
except Exception as e:
    print(type(e).__name__, e)

Temps : 0.26 s — 50000 lignes
Commandes par statut :
statut
livrée       38890
annulée       4568
en_cours      4058
retournée     2484
Name: count, dtype: int64

Commandes par canal :
canal
mobile_app    32519
web           17481
Name: count, dtype: int64

Type de montant_total_fcfa : str
Somme : 1905003220040500400035500348500450010115006020001100009550015800200007850025700571008950035000270008300058000234500 FCFA1510001400002150024750070600065003300042500445001790003050019500463500120200207500625008348002185003700083100034000158400017000900000121110012650049000165500266500117000104800013000139000216000224000390000250500467700655007000520005500055000109500425003998001320007900046940060900182000430004050051400238500530000325001690003500603005310006825001099005220007550053500060006850023000036160043050024100015000 FCFA95002274003600010600099700109500391300125001500009500048350036500145003640036000455004100031500274000819600458000655001115001000022200012300786800300052500128300038650069

**Question 2.a** — Remplissez la ligne `orders.csv` du tableau de relevés
(lignes, temps, mémoire deep, taille disque).

In [9]:
# === À COMPLÉTER ===
mem_Mo = orders.memory_usage(deep=True).sum() / 1024**2
disque_Mo = os.path.getsize(os.path.join(DATA_DIR, "orders.csv")) / 1024**2

releves["orders.csv"] = {"lignes": len(orders), "temps_s": round(t1 - t0, 2),
                            "memoire_Mo": round(mem_Mo, 1), "disque_Mo": round(disque_Mo, 1)}
releves["orders.csv"]

{'lignes': 50000,
 'temps_s': 0.26,
 'memoire_Mo': np.float64(17.7),
 'disque_Mo': 3.2}

## Exercice 3 — Jointure `orders` × `order_items`

Les jointures sont au cœur de l'analytique — et elles **coûtent cher**.
1. Chargez `order_items.csv`.
2. Mesurez la mémoire totale **avant** la jointure (somme des deux DataFrames).
3. Réalisez la jointure (`merge` sur `order_id`), chronométrez-la.
4. Mesurez la mémoire du résultat. Conclusion ?

In [10]:
# === À COMPLÉTER ===
items = pd.read_csv(os.path.join(DATA_DIR, "order_items.csv"))  # charger order_items.csv
mem_avant = orders.memory_usage(deep=True).sum() / 1024**2 + items.memory_usage(deep=True).sum() / 1024**2  # mémoire orders + items (Mo, deep)

t0 = time.perf_counter()
joint = orders.merge(items, on="order_id")  # merge orders x items sur order_id
t1 = time.perf_counter()

mem_joint = joint.memory_usage(deep=True).sum() / 1024**2  # mémoire du résultat (Mo, deep)
orders_disk = os.path.getsize(os.path.join(DATA_DIR, "orders.csv")) / 1024**2
items_disk = os.path.getsize(os.path.join(DATA_DIR, "order_items.csv")) / 1024**2
releves["orders+items (jointure)"] = {"lignes": len(joint), "temps_s": round(t1 - t0, 2),
                            "memoire_Mo": round(mem_joint, 1), "disque_Mo": round(orders_disk + items_disk, 1)}
releves["orders+items (jointure)"]

print("Jointure : %.2f s — %d lignes" % (t1 - t0, len(joint)))
print("Mémoire avant : %.0f Mo — résultat seul : %.0f Mo" % (mem_avant, mem_joint))

Jointure : 0.06 s — 112750 lignes
Mémoire avant : 38 Mo — résultat seul : 54 Mo


## Exercice 4 — `events.json` : le gros morceau

`events.json` est au format **JSON Lines** (un objet JSON par ligne).
À l'échelle 0.1 il reste chargeable (~330 000 lignes) ; mesurez précisément :
temps, mémoire deep, taille disque, et le **ratio mémoire/disque**.

In [11]:
# === À COMPLÉTER ===
t0 = time.perf_counter()
events = pd.read_json(os.path.join(DATA_DIR, "events.json"), lines=True)
t1 = time.perf_counter()

mem_events = events.memory_usage(deep=True).sum() / 1024**2
disque_events = os.path.getsize(os.path.join(DATA_DIR, "events.json")) / 1024**2

releves["events.json"] = {"lignes": len(events), "temps_s": round(t1 - t0, 2),
                            "memoire_Mo": round(mem_events, 1), "disque_Mo": round(disque_events, 1)}
releves["events.json"]

{'lignes': 329976,
 'temps_s': 4.44,
 'memoire_Mo': np.float64(137.7),
 'disque_Mo': 65.3}

## Exercice 5 — Synthèse des relevés (échelle 0.1)

In [12]:
pd.DataFrame(releves).T

,lignes,temps_s,memoire_Mo,disque_Mo
customers.csv,5025.0,0.09,2.9,0.6
orders.csv,50000.0,0.26,17.7,3.2
orders+items (jointure),112750.0,0.06,53.7,6.9
events.json,329976.0,4.44,137.7,65.3


## Partie E — Montée en charge : échelle 1.0 sur Colab

**À faire sur Google Colab** (pas sur votre laptop) :

1. Téléversez `generate_data.py` et `requirements.txt` dans la session ;
2. `!pip install -r requirements.txt` puis
   `!python3 generate_data.py --scale 1.0 --outdir ./data` (~2 min, 860 Mo) ;
3. Ré-exécutez les exercices 2 à 4 sur ces données (adaptez `DATA_DIR`) ;
4. Tentez le chargement intégral de `events.json` (≈ 685 Mo, 3,3 M lignes)
   et **observez** : durée, occupation RAM (panneau « RAM » de Colab),
   avertissements, voire crash du noyau.

> Un crash du noyau **est un résultat de mesure**, pas un échec : notez
> précisément ce qui s'est passé et à quel moment.

In [ ]:
# Cellule Colab (échelle 1.0) — recopiez vos mesures ci-dessous en local
releves_colab = {
    "orders.csv (1.0)"   : {"lignes": None, "temps_s": None, "memoire_Mo": None},
    "events.json (1.0)"  : {"lignes": None, "temps_s": None, "memoire_Mo": None,
                            "issue": "chargé / averti / crash ?"},
}
releves_colab

### Extrapolation

Complétez à partir de vos deux points de mesure (0.1 et 1.0), en supposant
d'abord une croissance **linéaire** :

| Scénario | `events.json` | Temps estimé | Mémoire estimée | Tenable sur 1 machine ? |
|---|---|---|---|---|
| TP échelle 0.1 (mesuré) | ≈ 70 Mo | … | … | … |
| TP échelle 1.0 (mesuré) | ≈ 685 Mo | … | … | … |
| Plateforme réelle, 1 an | ≈ 50 Go | … | … | … |
| Grand acteur régional | ≈ 1 To | … | … | … |

**Questions** (réponses dans `DIAGNOSTIC.md`) :
1. L'extrapolation linéaire est-elle optimiste ou pessimiste ? Pourquoi ?
2. Quelles parades *sans* distribution connaissez-vous (chunks, `dtype`
   optimisés, échantillonnage, formats binaires) — et jusqu'où
   repoussent-elles le mur ?
3. À partir de quel point le **scale-out** devient-il inévitable ?

## 6. Anomalies repérées (sans les corriger !)

Listez ici, **avec une preuve chiffrée** (un comptage, un exemple), les
bizarreries rencontrées. Elles seront traitées en séances 3 et 9.

In [ ]:
# === À COMPLÉTER ===
# Exemples de pistes (à vérifier et chiffrer) :
# - emails vides ou "N/A" dans customers
# - villes en casse/accents incohérents
# - dtype de orders["montant_total_fcfa"]
# - dates de naissance aberrantes
...

## 7. Vers le diagnostic

Avant de fermer ce notebook, vérifiez que vous savez répondre, **mesures à
l'appui**, aux trois questions du canevas `docs/DIAGNOSTIC.md` :

1. **Constats** — où, précisément, le traitement local a-t-il atteint ses
   limites (chiffres) ?
2. **Analyse** — pourquoi (mémoire vs disque, jointures, croissance des
   volumes) ?
3. **Besoins** — qu'attend-on d'une architecture distribuée (et que
   garde-t-on de Pandas) ?

Puis : `git add notebooks/ docs/ && git commit -m "TP1 : exploration et
diagnostic" && git push`.